# Immune Data Tokenization

## Convert to HF dataset


In [1]:
import sys
sys.path.append("..")

from models.scimmune.tokenizer import ScImmuneTokenizer # refactored version

import scanpy as sc
from pathlib import Path

import numpy as np
import torch
from tqdm import tqdm
from scipy.sparse import issparse

/home/s5srinivasan/immune-foundational-model/.venv/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
## Set folder paths
DATA_PATH = Path("../data/cellxgene_data")
ONTOLOGY_PATH = Path("../data/ontologies")
MODEL_PATH = Path("../models/scimmune")
UTIL_PATH = Path("../utils")
VOCAB_FILE = MODEL_PATH / "vocab_w_cell_type.json"

In [15]:
## Load dataset
adata_immune = sc.read_h5ad(DATA_PATH / "immune_900K_filtered.h5ad")

/home/s5srinivasan/immune-foundational-model/.venv/lib64/python3.9/site-packages/anndata/_core/anndata.py:1818: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [21]:
adata_test = sc.pp.subsample(adata_immune, n_obs=1000, copy=True)


In [22]:
tokenizer = ScImmuneTokenizer(vocab_file=str(VOCAB_FILE)) # initialize tokenizer
len(tokenizer)

60838

In [58]:
import numpy as np
import torch
from tqdm import tqdm

# Define the fields from obs to be turned into metadata tokens

metadata_fields = [
                    "cell_type_ontology_term_id",
                    # "self_reported_ethnicity_ontology_term_id", 
                    # "tissue_general_ontology_term_id",
                    # "development_stage_ontology_term_id",
                    # "sex_ontology_term_id",
                    # "disease_ontology_term_id"
]

gene_names = adata_test.var.feature_name.values # get gene names

tokenized_input_ids = []
tokenized_values = []

for i in tqdm(range(adata_test.n_obs)):
    
    # 1. Get dense expression vector
    row = adata_test.X[i]
    if not isinstance(row, np.ndarray):
        row = row.toarray().squeeze()

    # 2. Get metadata tokens
    obs_row = adata_test.obs.iloc[i]
    metadata_tokens = []
    for field in metadata_fields:
        val = obs_row.get(field)
        if isinstance(val, str) and val != "NA" and "=" not in val:
            token = f"<{field.split('_ontology_term_id')[0]}={val}>"
            metadata_tokens.append(token)

    # 3. Tokenize
    tokenized = tokenizer.tokenize_cell_batch(
        data=np.expand_dims(row, axis=0),
        gene_ids=gene_names,
        metadata_tokens=metadata_tokens,
        append_cls=True,
        include_zero_gene=False
    )
    
    input_ids, values = tokenized[0]
    tokenized_input_ids.append(input_ids)
    tokenized_values.append(values)

100%|██████████| 1000/1000 [00:04<00:00, 239.13it/s]


In [16]:
import torch

# Define the CLS token ID
CLS_TOKEN_ID = 60695
cls_token_tensor = torch.tensor([CLS_TOKEN_ID], dtype=tokenized_input_ids[0].dtype)

# Reorder tokens and values to ensure CLS token is at index 0
for i in range(len(tokenized_input_ids)):
    if tokenized_input_ids[i][1] == CLS_TOKEN_ID:
        # Swap the CLS token and its value to the first position
        tokenized_input_ids[i] = torch.cat(
            [cls_token_tensor, tokenized_input_ids[i][0].unsqueeze(0), tokenized_input_ids[i][2:]]
        )
        tokenized_values[i] = torch.cat(
            [tokenized_values[i][1].unsqueeze(0), tokenized_values[i][0].unsqueeze(0), tokenized_values[i][2:]]
        )

# Convert to Hugging Face dataset
from datasets import Dataset

hf_dataset = Dataset.from_dict({
    "genes": tokenized_input_ids,
    "values": tokenized_values
})

# Set dataset format for PyTorch
hf_dataset.set_format(type="torch", columns=["genes", "values"])


In [19]:
hf_dataset[400]

{'genes': tensor([60695, 60703, 30311, 30300, 19884, 31106,  4389, 35660, 32366, 13220,
         33303, 10193, 31742, 35628, 30415, 30942, 16598, 19646,  8743, 30342,
         30453, 34714, 19767, 16987, 30567, 30388, 10830, 21387, 34528, 20433,
          2626, 20664, 17652,  9236, 33889,  2553, 30414, 32821, 19282, 13217,
         20703, 31034, 13278, 20562, 30389, 34893, 19208, 31945, 33777, 16264,
         30324,  7496,  7388, 17348,  7371, 21350, 16234, 19285,  5008, 31543,
          5136,  8248, 30334, 20633,  2575, 33319, 30325, 30305, 32582, 34837,
         30417, 30304, 30590, 33443,  2558, 16131, 30321,  4185,  9466, 30315,
         17712, 10683, 21524,  4284, 12933,  8374, 20115, 30329,  2844,  9791,
          8957, 30348,  9418, 19689, 31748,  7862, 10717, 30326, 36071, 30395,
         11210, 11074,  7884, 12051,  9705, 30333, 30952, 30384,  5454, 15701,
         11008, 31260, 30373, 17975, 11003, 20880, 11020, 17275, 11047, 11032,
         11031, 15910, 15907, 31915, 10417,

In [22]:
hf_dataset.save_to_disk(MODEL_PATH.joinpath("data/tokenized"))

Saving the dataset (38/38 shards): 100%|██████████| 895382/895382 [00:06<00:00, 133360.59 examples/s]
